# Cardiovascular Disease Prediction

**Author:** Sumaiya Ahmed  
**Dataset:** Cardiovascular Disease Dataset (70,000 patient records)  
**Objective:** Predict the presence of cardiovascular disease using patient clinical and lifestyle data, and evaluate the performance of machine learning classifiers.

---

## Project Overview

Cardiovascular disease (CVD) is the leading cause of mortality globally, accounting for approximately 17.9 million deaths per year (WHO, 2021). Early identification of at-risk individuals through data-driven approaches offers significant clinical utility. This project applies exploratory data analysis (EDA) and supervised machine learning classification to a dataset of 70,000 patient records, examining the predictive value of clinical features such as blood pressure, cholesterol, glucose, BMI, and lifestyle variables.

**Target variable:** `cardio` (0 = no CVD, 1 = CVD present)

**Models evaluated:**
- Random Forest Classifier
- Decision Tree Classifier

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Set consistent plot aesthetics
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

print('All libraries imported successfully.')

## 2. Load and Inspect the Data

In [ ]:
# Load dataset — note the semicolon delimiter used in this dataset
df = pd.read_csv('cardio_train.csv', sep=';')

# Drop the 'id' column — it is a row identifier with no predictive value
df = df.drop(columns=['id'])

print(f'Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head()

In [ ]:
# Summary statistics — identifies range, mean, and distribution of each feature
df.describe().round(2)

In [ ]:
# Check data types and confirm no missing values
print('Data types:')
print(df.dtypes)
print(f'\nMissing values: {df.isnull().sum().sum()}')
print(f'Duplicate rows: {df.duplicated().sum()}')

## 3. Data Cleaning

Blood pressure values (`ap_hi` = systolic, `ap_lo` = diastolic) can contain implausible entries due to data entry errors. Clinically, a systolic pressure below 80 mmHg or above 250 mmHg, and a diastolic below 60 or above 200 mmHg, are physiologically implausible and should be removed. Additionally, height and weight values at extreme ends of the distribution are filtered.

In [ ]:
rows_before = len(df)

# Remove physiologically implausible blood pressure readings
df = df[(df['ap_hi'] >= 80) & (df['ap_hi'] <= 250)]
df = df[(df['ap_lo'] >= 60) & (df['ap_lo'] <= 200)]

# Ensure diastolic pressure does not exceed systolic (physiologically impossible)
df = df[df['ap_hi'] >= df['ap_lo']]

# Remove implausible height and weight values
df = df[(df['height'] >= 140) & (df['height'] <= 210)]
df = df[(df['weight'] >= 30) & (df['weight'] <= 200)]

rows_after = len(df)
print(f'Rows removed during cleaning: {rows_before - rows_after:,}')
print(f'Rows retained: {rows_after:,}')

## 4. Feature Engineering

Age is recorded in days. Converting to years produces a more clinically interpretable and visually informative feature. BMI (Body Mass Index) is derived from height and weight and is a recognised cardiovascular risk factor.

In [ ]:
# Convert age from days to years (rounded to nearest integer)
df['age_years'] = (df['age'] / 365).round(0).astype(int)

# Derive BMI: weight (kg) / height (m)^2
df['bmi'] = (df['weight'] / ((df['height'] / 100) ** 2)).round(1)

# Drop the original age-in-days column as age_years is more interpretable
df = df.drop(columns=['age'])

print('New features added: age_years, bmi')
print(f'Age range: {df["age_years"].min()} to {df["age_years"].max()} years')
print(f'BMI range: {df["bmi"].min()} to {df["bmi"].max()}')

## 5. Exploratory Data Analysis (EDA)

In [ ]:
# --- Target variable distribution ---
fig, ax = plt.subplots()
counts = df['cardio'].value_counts()
ax.bar(['No CVD (0)', 'CVD Present (1)'], counts.values, color=['#5DCAA5', '#D85A30'], edgecolor='black')
ax.set_title('Distribution of Cardiovascular Disease Diagnosis', fontsize=13)
ax.set_ylabel('Patient Count')
for i, v in enumerate(counts.values):
    ax.text(i, v + 200, f'{v:,}', ha='center', fontsize=11)
plt.tight_layout()
plt.show()
print(f'Class balance: {(counts[1]/len(df)*100):.1f}% CVD positive')

In [ ]:
# --- Age distribution by CVD status ---
fig, ax = plt.subplots()
for label, grp in df.groupby('cardio'):
    ax.hist(grp['age_years'], bins=20, alpha=0.6,
            label='CVD Present' if label == 1 else 'No CVD',
            edgecolor='black')
ax.set_title('Age Distribution by Cardiovascular Disease Status', fontsize=13)
ax.set_xlabel('Age (years)')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Gender vs CVD status ---
fig, ax = plt.subplots()
sns.countplot(x='gender', hue='cardio', data=df,
              palette='colorblind', edgecolor='black', ax=ax)
ax.set_title('CVD Prevalence by Gender', fontsize=13)
ax.set_xlabel('Gender (1 = Female, 2 = Male)')
ax.set_ylabel('Count')
ax.legend(title='CVD', labels=['No CVD', 'CVD Present'])
plt.tight_layout()
plt.show()

In [ ]:
# --- Cholesterol and Glucose levels by CVD status ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, col, title in zip(axes,
                           ['cholesterol', 'gluc'],
                           ['Cholesterol Level', 'Glucose Level']):
    sns.countplot(x=col, hue='cardio', data=df,
                  palette='colorblind', edgecolor='black', ax=ax)
    ax.set_title(f'{title} by CVD Status', fontsize=12)
    ax.set_xlabel(f'{title} (1=Normal, 2=Above Normal, 3=Well Above Normal)')
    ax.set_ylabel('Count')
    ax.legend(title='CVD', labels=['No CVD', 'CVD Present'])

plt.tight_layout()
plt.show()

In [ ]:
# --- Correlation heatmap ---
plt.figure(figsize=(12, 8))
corr_matrix = df.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # Show lower triangle only
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', linewidths=0.5, annot_kws={'size': 9})
plt.title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Model Building

The dataset is split 70/30 into training and test sets. Two classifiers are evaluated: a **Random Forest Classifier** (an ensemble of decision trees that reduces overfitting via bagging) and a **Decision Tree Classifier** (a single tree that is faster but more prone to overfitting). Both are evaluated against the held-out test set.

In [ ]:
# Define features (X) and target (y)
X = df.drop(columns=['cardio'])
y = df['cardio']

# Split into training and test sets (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]:,} samples')
print(f'Test set:     {X_test.shape[0]:,} samples')

In [ ]:
# --- Random Forest Classifier ---
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

rf_accuracy = accuracy_score(y_test, rf_preds)
print(f'Random Forest Accuracy: {rf_accuracy:.4f} ({rf_accuracy*100:.2f}%)')
print('\nClassification Report:')
print(classification_report(y_test, rf_preds, target_names=['No CVD', 'CVD Present']))

In [ ]:
# --- Decision Tree Classifier ---
dt_model = DecisionTreeClassifier(max_depth=8, random_state=42)
dt_model.fit(X_train, y_train)
dt_preds = dt_model.predict(X_test)

dt_accuracy = accuracy_score(y_test, dt_preds)
print(f'Decision Tree Accuracy: {dt_accuracy:.4f} ({dt_accuracy*100:.2f}%)')
print('\nClassification Report:')
print(classification_report(y_test, dt_preds, target_names=['No CVD', 'CVD Present']))

In [ ]:
# --- Confusion matrices side by side ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, preds, title in zip(
    axes,
    [rf_preds, dt_preds],
    ['Random Forest', 'Decision Tree']
):
    cm = confusion_matrix(y_test, preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                   display_labels=['No CVD', 'CVD Present'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{title} — Confusion Matrix', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# --- Feature importance (Random Forest) ---
importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=importance_df,
            palette='muted', edgecolor='black')
plt.title('Feature Importance — Random Forest Classifier', fontsize=13)
plt.xlabel('Relative Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

print(importance_df.to_string(index=False))

## 7. Model Comparison and Conclusions

In [ ]:
# Summary comparison table
comparison = pd.DataFrame({
    'Model': ['Random Forest', 'Decision Tree'],
    'Accuracy': [rf_accuracy, dt_accuracy],
    'Accuracy (%)': [f'{rf_accuracy*100:.2f}%', f'{dt_accuracy*100:.2f}%']
})

print('--- Model Performance Summary ---')
print(comparison.to_string(index=False))

best = 'Random Forest' if rf_accuracy >= dt_accuracy else 'Decision Tree'
print(f'\nBest performing model: {best}')

## Conclusions

This analysis demonstrates the application of machine learning classification to cardiovascular disease prediction using clinical and lifestyle data from 70,000 patients.

**Key findings:**
- Systolic blood pressure (`ap_hi`), age, and BMI emerged as the strongest predictors of CVD diagnosis.
- Elevated cholesterol and glucose levels showed a clear association with positive CVD diagnoses.
- The Random Forest Classifier outperformed the Decision Tree, consistent with its ensemble design reducing variance and overfitting.
- The dataset was broadly balanced (~50% positive / ~50% negative), which reduces the risk of misleading accuracy metrics.

**Limitations:**
- CVD diagnosis is self-reported in this dataset, introducing potential misclassification bias.
- No cross-validation was applied in this analysis; future iterations should use k-fold CV for more robust performance estimates.
- Hyperparameter tuning (e.g., via GridSearchCV) would likely improve model performance further.

**Potential extensions:**
- Logistic Regression baseline for interpretability
- ROC-AUC curve comparison across models
- SHAP values for clinical interpretability of feature contributions